# FREUID Challenge 2026 (IJCAI-ECAI) — Notebook v1

**Task:** Binary fraud detection on identity document images  
**Metric:** FREUID Score (lower is better) = 1 − harmonic-mean(g_audet, g_apcer)  
**Key constraint:** Must perform well at 1% BPCER operating point  
**Output:** `id,score` where score ∈ [0,1], higher = more likely fraudulent  
**Data:** `/kaggle/input/datasets/mahiux/freuid-data/`  
**Weights:** `/kaggle/input/models/timm/tf-efficientnet/pytorch/tf-efficientnet-b4/1/`

---
## Notebook Structure
1. Environment Setup  
2. Config  
3. Data Loading & EDA  
4. FREUID Metric Implementation  
5. Cross-Validation Splits  
6. Augmentation Pipeline  
7. Dataset Class  
8. Model Architecture  
9. Loss Functions  
10. Training Pipeline  
11. Score Calibration  
12. Inference & Submission  
13. Ensemble Utilities  
14. Visualization & Ablation Logger  
15. Quick-Start Execution Cells  

## 1. Environment Setup

In [ ]:
import os
import time
import random
import warnings
import math
import json
import shutil
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List

# ── Must be set BEFORE importing albumentations and timm ─────────────────────
# Kaggle's sandboxed network blocks outbound DNS to huggingface.co and PyPI.
# These env vars disable all telemetry and version-check network calls.
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'   # suppress albumentations version check
os.environ['HF_HUB_OFFLINE']           = '1'   # force HuggingFace hub offline mode
os.environ['TRANSFORMERS_OFFLINE']     = '1'   # belt-and-suspenders for transformers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import scipy.stats
import scipy.optimize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42

def seed_everything(seed: int = SEED) -> None:
    """Seed all RNGs for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__}')
print(f'NumPy  : {np.__version__}')
print(f'timm   : {timm.__version__}')

## 2. Config

In [ ]:
@dataclass
class CFG:
    # ── Paths ─────────────────────────────────────────────────────────────────
    # Dataset: uploaded as personal Kaggle dataset (mahiux/freuid-data)
    # Weights: Kaggle model hub (timm/tf-efficientnet)
    #
    # Toggle USE_FULL_DATA to switch between full release and sample data
    # without changing anything else in the notebook.
    #
    # CRITICAL: private test set includes 2 document types UNSEEN in train
    # or public_test — the model must not over-rely on type embedding.
    DATA_DIR:          str   = '/kaggle/input/datasets/maheshwarmishra/freuid-data'
    WEIGHTS_PATH:      str   = ('/kaggle/input/models/timm/tf-efficientnet/'
                                'pytorch/tf-efficientnet-b4/1/'
                                'tf_efficientnet_b4_aa-818f208c.pth')
    USE_FULL_DATA:     bool  = True
    OUTPUT_DIR:        str   = '/kaggle/working'
    CHECKPOINT_DIR:    str   = '/kaggle/working/checkpoints'

    # Derived path fields — resolved in __post_init__
    TRAIN_LABELS_FILE: str   = ''
    TRAIN_IMG_SUBDIR:  str   = ''
    TEST_IMG_SUBDIR:   str   = 'public_test'

    # ── Model ─────────────────────────────────────────────────────────────────
    # tf_efficientnet_b4: matches the Kaggle-hosted weights file.
    # Apache-2.0 via timm — OSI-compatible, prize-eligible.
    BACKBONE:          str   = 'tf_efficientnet_b4'
    PRETRAINED:        bool  = True
    IMG_SIZE:          int   = 224   # set to 384 for final competition run
    DROP_RATE:         float = 0.3
    USE_METADATA:      bool  = True
    N_DOC_TYPES:       int   = 256   # upper bound; updated after vocab build
    DOC_EMB_DIM:       int   = 16

    # ── Training ──────────────────────────────────────────────────────────────
    N_FOLDS:           int   = 5
    EPOCHS:            int   = 20
    BATCH_SIZE:        int   = 32
    GRAD_ACCUM:        int   = 1    # set to 2 for effective batch=64 at final run
    NUM_WORKERS:       int   = 0    # 0 avoids DataLoader deadlock on Kaggle
    PIN_MEMORY:        bool  = True
    SAMPLER_CAP:       int   = 8000  # max samples per epoch; set to -1 for full

    # ── Optimizer ─────────────────────────────────────────────────────────────
    LR:                float = 2e-4
    LR_MIN:            float = 1e-6
    WEIGHT_DECAY:      float = 1e-4
    WARMUP_EPOCHS:     int   = 2
    GRAD_CLIP:         float = 1.0

    # ── Loss ──────────────────────────────────────────────────────────────────
    # 'focal'      → class-imbalance-aware focal loss
    # 'pauc_focal' → pAUC in [0, FPR_MAX] + focal blend (best for FREUID metric)
    # 'bce'        → standard binary cross-entropy
    LOSS_TYPE:         str   = 'pauc_focal'
    FOCAL_GAMMA:       float = 2.0
    FOCAL_ALPHA:       float = 0.75   # weight for positive (fraud) class
    FPR_MAX:           float = 0.10   # pAUC region: optimize up to 10% FPR
    PAUC_WEIGHT:       float = 0.5    # blend: (1-w)*focal + w*pauc

    # ── Augmentation ──────────────────────────────────────────────────────────
    # These simulate print-and-capture ('analog hole') artifacts.
    # Reduced for fast validation run — restore to 0.5/0.3/0.3/0.2/0.15
    # for the final competition training run.
    AUG_P_JPEG:        float = 0.2
    AUG_P_NOISE:       float = 0.1
    AUG_P_BLUR:        float = 0.1
    AUG_P_MOIRE:       float = 0.05
    AUG_P_WARP:        float = 0.05

    # ── Calibration ───────────────────────────────────────────────────────────
    # Mandatory: APCER@1%BPCER depends on score distribution near a fixed
    # threshold, not just global ranking — calibration directly affects score.
    CALIBRATE:         bool  = True
    CALIB_METHOD:      str   = 'temperature'   # 'temperature' | 'isotonic'

    # ── Mixed precision ───────────────────────────────────────────────────────
    AMP:               bool  = True

    # ── W&B experiment tracking ───────────────────────────────────────────────
    USE_WANDB:         bool  = False
    WANDB_PROJECT:     str   = 'freuid-2026'
    WANDB_RUN_NAME:    str   = 'effb4-pauc-v1'

    # ── Evaluation ────────────────────────────────────────────────────────────
    BPCER_TARGET:      float = 0.01   # fixed 1% BPCER operating point

    def __post_init__(self):
        """Resolve fields whose values depend on USE_FULL_DATA."""
        if not self.TRAIN_LABELS_FILE:
            self.TRAIN_LABELS_FILE = (
                'train_labels.csv' if self.USE_FULL_DATA
                else 'train_sample_labels.csv'
            )
        if not self.TRAIN_IMG_SUBDIR:
            self.TRAIN_IMG_SUBDIR = 'train' if self.USE_FULL_DATA else 'train_sample'


CFG = CFG()
Path(CFG.CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path(CFG.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print('Config loaded.')
print(f'  USE_FULL_DATA    : {CFG.USE_FULL_DATA}')
print(f'  TRAIN_LABELS_FILE: {CFG.TRAIN_LABELS_FILE}')
print(f'  TRAIN_IMG_SUBDIR : {CFG.TRAIN_IMG_SUBDIR}')
print(f'  TEST_IMG_SUBDIR  : {CFG.TEST_IMG_SUBDIR}')
print(f'  BACKBONE         : {CFG.BACKBONE}')
print(f'  IMG_SIZE         : {CFG.IMG_SIZE}')
print(f'  SAMPLER_CAP      : {CFG.SAMPLER_CAP}')
print(f'  LOSS_TYPE        : {CFG.LOSS_TYPE}')

## 3. Data Loading & EDA

In [ ]:
data_dir = Path(CFG.DATA_DIR)

# ── Train labels ──────────────────────────────────────────────────────────────
labels_path = data_dir / CFG.TRAIN_LABELS_FILE
if not labels_path.exists():
    raise FileNotFoundError(
        f'Labels file not found: {labels_path}\n'
        f'Check CFG.DATA_DIR and CFG.USE_FULL_DATA (currently {CFG.USE_FULL_DATA}).'
    )
train_df = pd.read_csv(labels_path)

# Normalize is_digital: CSV stores True/False — convert to int {0, 1}
train_df['is_digital'] = train_df['is_digital'].astype(bool).astype(int)

# Validate expected columns are present
required = {'id', 'image_path', 'label', 'is_digital', 'type'}
missing  = required - set(train_df.columns)
if missing:
    raise ValueError(f'train_df missing columns: {missing}. Found: {train_df.columns.tolist()}')

# ── Test set (public_test) ────────────────────────────────────────────────────
# public_test has NO companion CSV — it is a flat folder of bare-UUID .jpeg
# files (the true unlabeled holdout). Build test_df by listing the folder.
test_img_dir = data_dir / CFG.TEST_IMG_SUBDIR
if test_img_dir.exists():
    test_files = sorted(test_img_dir.glob('*.jpeg')) + sorted(test_img_dir.glob('*.jpg'))
    test_df = pd.DataFrame({
        'id':         [p.stem for p in test_files],
        'image_path': [f'{CFG.TEST_IMG_SUBDIR}/{p.name}' for p in test_files],
        'is_digital': 0,    # unknown at test time; defaults to recaptured
        'type_idx':   0,    # unknown type → <UNK> embedding (index 0)
    })
    print(f'Test images found: {len(test_df)}')
else:
    print(f'WARNING: {test_img_dir} not found — test_df will be empty.')
    test_df = pd.DataFrame(columns=['id', 'image_path', 'is_digital', 'type_idx'])

# ── Sample submission ─────────────────────────────────────────────────────────
sub_csv = data_dir / 'sample_submission.csv'
sub_df  = pd.read_csv(sub_csv) if sub_csv.exists() else pd.DataFrame()

print(f'Train shape : {train_df.shape}')
print(f'Test shape  : {test_df.shape}')
print(f'Fraud rate  : {train_df["label"].mean():.3f}')
print(f'Columns     : {train_df.columns.tolist()}')
train_df.head()

In [ ]:
# ── EDA visualizations ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

train_df['label'].value_counts().plot(
    kind='bar', ax=axes[0], color=['#2196F3', '#F44336'])
axes[0].set_title('Label Distribution')
axes[0].set_xticklabels(['Genuine (0)', 'Fraud (1)'], rotation=0)

pd.crosstab(train_df['is_digital'], train_df['label']).plot(
    kind='bar', ax=axes[1], color=['#2196F3', '#F44336'])
axes[1].set_title('is_digital × label')
axes[1].set_xticklabels(['Recaptured (0)', 'Digital (1)'], rotation=0)
axes[1].legend(['Genuine', 'Fraud'])

train_df['type'].value_counts().head(15).plot(
    kind='barh', ax=axes[2], color='#9C27B0')
axes[2].set_title('Top-15 Document Types')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n=== Label distribution ===')
print(train_df['label'].value_counts())
print('\n=== is_digital distribution ===')
print(train_df['is_digital'].value_counts())
print('\n=== Cross-tab: label × is_digital ===')
print(pd.crosstab(train_df['label'], train_df['is_digital']))
print(f'\nUnique document types: {train_df["type"].nunique()}')
print(train_df['type'].value_counts().head(20))

In [ ]:
# ── Document type → integer index ─────────────────────────────────────────────
# Build vocab on train only. public_test has no type column.
# IMPORTANT: private test set includes 2 types UNSEEN in train/public_test.
# They will receive <UNK> (index 0) at inference time — this is intentional.
all_types = train_df['type'].unique()
type2idx  = {t: i + 1 for i, t in enumerate(sorted(all_types))}  # 0 = <UNK>
type2idx['<UNK>'] = 0

train_df['type_idx'] = train_df['type'].map(type2idx).fillna(0).astype(int)

# test_df has no type column — assign 0 (<UNK>) for all
if 'type_idx' not in test_df.columns:
    test_df['type_idx'] = 0

CFG.N_DOC_TYPES = len(type2idx) + 1
print(f'Document type vocabulary size: {CFG.N_DOC_TYPES}')
print(f'Sample type mappings: { {k: v for k, v in list(type2idx.items())[:5]} }')

In [ ]:
# ── Image path resolver ───────────────────────────────────────────────────────
# Handles the doubled-folder quirk seen in some Kaggle dataset mirrors
# (e.g. train/train/<id>.jpg on disk vs. train/<id>.jpg in the CSV).
def resolve_image_path(base_dir: Path, rel_path: str) -> Path:
    candidate = base_dir / rel_path
    if candidate.exists():
        return candidate
    parts = Path(rel_path).parts
    if len(parts) > 1:
        doubled = base_dir / parts[0] / rel_path
        if doubled.exists():
            return doubled
    flat = base_dir / Path(rel_path).name
    if flat.exists():
        return flat
    return candidate


# ── Sample image viewer ───────────────────────────────────────────────────────
def show_samples(df: pd.DataFrame, n: int = 8, title: str = '') -> None:
    """Display n random images with their labels and metadata."""
    sample = df.sample(n=min(n, len(df)), random_state=SEED)
    cols   = min(n, 4)
    rows   = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 5 * rows))
    for ax, (_, row) in zip(np.array(axes).flat, sample.iterrows()):
        img_path = resolve_image_path(data_dir, row['image_path'])
        try:
            ax.imshow(Image.open(img_path).convert('RGB'))
        except Exception:
            ax.text(0.5, 0.5, 'Load error', ha='center', va='center',
                    transform=ax.transAxes)
        lbl = 'FRAUD' if row.get('label', -1) == 1 else 'GENUINE'
        dig = 'digital' if row.get('is_digital', -1) == 1 else 'recaptured'
        ax.set_title(f"{lbl}\n{row.get('type', '?')} | {dig}", fontsize=8)
        ax.axis('off')
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


show_samples(train_df, n=8, title='Random Training Samples')

## 4. FREUID Metric Implementation

Full local implementation so every validation fold can be scored exactly as Kaggle would.  
**Never** use AUC-ROC alone as a proxy — the FREUID metric penalizes specifically at the 1% BPCER point.

- **BPCER** (FAR) = FP / (FP + TN) — genuine samples misclassified as fraud  
- **APCER** (FRR) = FN / (FN + TP) — attacks that evade detection  
- **AuDET** = area under DET curve (FAR vs FRR); 0 = perfect, 0.5 = random  
- **FREUID** = 1 − harmonic_mean(1−AuDET, 1−APCER@1%BPCER)

In [ ]:
def compute_det_curve(
    labels: np.ndarray,
    scores: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Sweep all unique decision thresholds and return (thresholds, FAR, FRR).

    Convention:
      label=1 → ATTACK (positive class)
      label=0 → BONA-FIDE (negative class)
      FAR  = BPCER = FP / (FP + TN)  genuine flagged as fraud
      FRR  = APCER = FN / (FN + TP)  attacks missed
    """
    thresholds = np.unique(scores)[::-1]   # high → low
    n_genuine  = (labels == 0).sum()
    n_attack   = (labels == 1).sum()
    assert n_genuine > 0 and n_attack > 0, \
        f'Need both classes. Got {n_genuine} genuine, {n_attack} attack.'

    far_list, frr_list = [], []
    for thresh in thresholds:
        preds = (scores >= thresh).astype(int)
        FP = ((preds == 1) & (labels == 0)).sum()
        FN = ((preds == 0) & (labels == 1)).sum()
        TP = ((preds == 1) & (labels == 1)).sum()
        TN = ((preds == 0) & (labels == 0)).sum()
        far_list.append(FP / (FP + TN) if (FP + TN) > 0 else 0.0)
        frr_list.append(FN / (FN + TP) if (FN + TP) > 0 else 0.0)

    return thresholds, np.array(far_list), np.array(frr_list)


def compute_audet(far: np.ndarray, frr: np.ndarray) -> float:
    """Area under the DET curve via trapezoidal integration. 0=perfect, 0.5=random."""
    order = np.argsort(far)
    far_s = np.concatenate([[0.0], far[order], [1.0]])
    frr_s = np.concatenate([[1.0], frr[order], [0.0]])
    # np.trapz removed in NumPy 2.0 — use np.trapezoid if available
    trapz = getattr(np, 'trapezoid', None) or np.trapz
    return float(trapz(frr_s, far_s))


def compute_apcer_at_bpcer(
    labels: np.ndarray,
    scores: np.ndarray,
    bpcer_target: float = 0.01,
) -> float:
    """
    APCER (FRR) at the operating point where BPCER (FAR) <= bpcer_target.
    Returns 1.0 (worst possible) if the model cannot achieve the target BPCER.
    """
    _, far, frr = compute_det_curve(labels, scores)
    valid = far <= bpcer_target
    return float(frr[valid].min()) if valid.any() else 1.0


def compute_freuid_score(
    labels: np.ndarray,
    scores: np.ndarray,
    bpcer_target: float = 0.01,
    eps: float = 1e-8,
) -> Dict[str, float]:
    """
    Compute the full FREUID Score and all intermediate components.

    Returns dict with: freuid, audet, apcer_1pct, g_audet, g_apcer, auc_roc
    """
    labels = np.asarray(labels)
    scores = np.asarray(scores)

    _, far, frr = compute_det_curve(labels, scores)
    audet       = compute_audet(far, frr)
    apcer_1pct  = compute_apcer_at_bpcer(labels, scores, bpcer_target)

    g_audet = 1.0 - audet
    g_apcer = 1.0 - apcer_1pct
    denom   = g_audet + g_apcer
    # Epsilon guard: if both components are 0, FREUID = 1 (worst)
    freuid  = 1.0 - (2.0 * g_audet * g_apcer) / denom if abs(denom) > eps else 1.0

    try:
        auc_roc = roc_auc_score(labels, scores)
    except Exception:
        auc_roc = float('nan')

    return {
        'freuid':     freuid,
        'audet':      audet,
        'apcer_1pct': apcer_1pct,
        'g_audet':    g_audet,
        'g_apcer':    g_apcer,
        'auc_roc':    auc_roc,
    }


def evaluate_breakdown(
    df: pd.DataFrame,
    scores: np.ndarray,
    bpcer_target: float = 0.01,
) -> pd.DataFrame:
    """
    Evaluate FREUID Score broken down by is_digital and document type.
    Use this to diagnose where the model fails — critical for competition.
    """
    df = df.reset_index(drop=True)
    rows = []

    # Overall
    m = compute_freuid_score(df['label'].values, scores, bpcer_target)
    rows.append({'group': 'OVERALL', 'subset': 'all', 'n': len(df), **m})

    # By is_digital
    for dig_val, dig_name in [(0, 'recaptured'), (1, 'digital')]:
        mask = (df['is_digital'] == dig_val).values
        if mask.sum() < 10 or len(np.unique(df['label'].values[mask])) < 2:
            continue
        m = compute_freuid_score(df['label'].values[mask], scores[mask], bpcer_target)
        rows.append({'group': 'is_digital', 'subset': dig_name,
                     'n': int(mask.sum()), **m})

    # By document type (only types with enough samples of both classes)
    for doc_type, grp in df.groupby('type'):
        if len(grp) < 20 or len(grp['label'].unique()) < 2:
            continue
        idx = grp.index.values
        m = compute_freuid_score(grp['label'].values, scores[idx], bpcer_target)
        rows.append({'group': 'doc_type', 'subset': doc_type,
                     'n': len(grp), **m})

    return pd.DataFrame(rows)


def plot_det_curve(
    labels: np.ndarray,
    scores: np.ndarray,
    title:  str = 'DET Curve',
) -> None:
    """Plot the Detection Error Trade-off curve with 1% BPCER operating point marked."""
    _, far, frr = compute_det_curve(labels, scores)
    audet = compute_audet(far, frr)
    apcer = compute_apcer_at_bpcer(labels, scores, CFG.BPCER_TARGET)
    order = np.argsort(far)

    plt.figure(figsize=(6, 6))
    plt.plot(far[order], frr[order], 'b-', lw=2, label=f'AuDET={audet:.4f}')
    plt.axvline(CFG.BPCER_TARGET, color='red', ls='--',
                label=f'BPCER={CFG.BPCER_TARGET*100:.0f}%')
    plt.scatter([CFG.BPCER_TARGET], [apcer], color='red', zorder=5,
                label=f'APCER@1%={apcer:.4f}')
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
    plt.fill_between(far[order], frr[order], alpha=0.08)
    plt.xlabel('FAR (BPCER)')
    plt.ylabel('FRR (APCER)')
    plt.title(title)
    plt.legend()
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(f'{CFG.OUTPUT_DIR}/det_curve_{title.replace(" ", "_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()


# ── Sanity checks ─────────────────────────────────────────────────────────────
np.random.seed(0)
_lbl = np.random.binomial(1, 0.5, 500)
_rnd = np.random.uniform(0, 1, 500)
_prf = _lbl.astype(float) + np.random.uniform(0, 0.01, 500)

m_rnd = compute_freuid_score(_lbl, _rnd)
m_prf = compute_freuid_score(_lbl, _prf)
print('Sanity check — random scores (expect FREUID ≈ 1.0, AUC ≈ 0.5):')
print(f'  FREUID={m_rnd["freuid"]:.4f}  AuDET={m_rnd["audet"]:.4f}  AUC={m_rnd["auc_roc"]:.4f}')
print('Sanity check — near-perfect scores (expect FREUID ≈ 0.0, AUC ≈ 1.0):')
print(f'  FREUID={m_prf["freuid"]:.4f}  AuDET={m_prf["audet"]:.4f}  AUC={m_prf["auc_roc"]:.4f}')

## 5. Cross-Validation Splits

In [ ]:
# Stratification key: label × is_digital × document_type
# Ensures every fold sees all attack modalities and document types.
# Prevents a fold from having e.g. all CROATIA/ID in train with none in val.
train_df['strat_key'] = (
    train_df['label'].astype(str) + '_' +
    train_df['is_digital'].astype(str) + '_' +
    train_df['type'].astype(str)
)

# Cap n_splits at the smallest stratum size (min 2) to avoid ValueError
# for rare document types with very few samples.
min_stratum  = train_df['strat_key'].value_counts().min()
n_folds_safe = max(2, min(CFG.N_FOLDS, min_stratum))
if n_folds_safe < CFG.N_FOLDS:
    print(f'Smallest stratum has {min_stratum} samples — '
          f'using n_splits={n_folds_safe} instead of CFG.N_FOLDS={CFG.N_FOLDS}.')

skf = StratifiedKFold(n_splits=n_folds_safe, shuffle=True, random_state=SEED)
train_df['fold'] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(train_df, train_df['strat_key'])):
    train_df.loc[train_df.index[val_idx], 'fold'] = fold_idx

assert (train_df['fold'] == -1).sum() == 0, 'Some rows were not assigned a fold!'
print(f'Folds created: {n_folds_safe}')
print(train_df.groupby('fold')['label'].value_counts().unstack())

## 6. Augmentation Pipeline

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def build_recapture_aug() -> A.Compose:
    """
    Physics-inspired augmentations simulating the print-and-capture 'analog hole':
    printing a document and re-photographing it with a camera.

    Purpose: teach the model that these degradations alone ≠ fraud, so it
    learns to detect semantic/structural inconsistencies instead of
    relying on fragile digital artifacts suppressed by recapture.

    Applied on top of standard spatial augmentations during training.
    Probabilities are reduced in CFG for fast validation — restore to
    0.5/0.3/0.3/0.2/0.15 for the final competition training run.
    """
    return A.Compose([
        # Lens / focus effects (camera optics variability)
        A.OneOf([
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.Defocus(radius=(1, 3), p=1.0),
        ], p=CFG.AUG_P_BLUR),

        # Sensor noise (camera ISO / gain)
        A.OneOf([
            A.GaussNoise(var_limit=(5, 30), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
        ], p=CFG.AUG_P_NOISE),

        # JPEG compression artifacts (camera on-chip compression)
        A.ImageCompression(quality_lower=40, quality_upper=85, p=CFG.AUG_P_JPEG),

        # Physical capture angle distortion
        A.Perspective(scale=(0.02, 0.08), p=CFG.AUG_P_WARP),

        # Ambient lighting variability
        A.RandomBrightnessContrast(
            brightness_limit=0.3, contrast_limit=0.3, p=0.4),
        A.HueSaturationValue(
            hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),

        # Moiré pattern simulation (overlapping raster grids: printer + display)
        A.GridDistortion(num_steps=4, distort_limit=0.08, p=CFG.AUG_P_MOIRE),
    ], p=1.0)


def build_train_transform(img_size: int = CFG.IMG_SIZE) -> A.Compose:
    """Full training augmentation pipeline."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05, scale_limit=0.1,
            rotate_limit=10, border_mode=0, p=0.5),
        A.CoarseDropout(
            max_holes=4, max_height=img_size // 8,
            max_width=img_size // 8, fill_value=0, p=0.3),
        build_recapture_aug(),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def build_val_transform(img_size: int = CFG.IMG_SIZE) -> A.Compose:
    """Minimal validation/inference transform: resize + normalize only."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


print('Augmentation pipelines defined.')
print('  build_train_transform(): full pipeline with recapture simulation')
print('  build_val_transform()  : resize + normalize only (fast, no randomness)')

## 7. Dataset Class & Weighted Sampler

In [ ]:
class FREUIDDataset(Dataset):
    """
    Document fraud detection dataset.

    Returns a dict per sample:
      image      : (C, H, W) float tensor  — augmented/normalized
      is_digital : scalar float tensor {0.0, 1.0}  — modality metadata
      type_idx   : scalar long tensor  — document type index (0=<UNK>)
      id         : str  — for submission alignment
      label      : scalar long tensor {0, 1}  — only present if is_train=True

    Path resolution handles the doubled-folder quirk seen in some
    Kaggle dataset mirrors (e.g. train/train/<id>.jpg on disk vs.
    train/<id>.jpg in the CSV image_path column).
    """

    def __init__(
        self,
        df:        pd.DataFrame,
        data_dir:  Path,
        transform: Optional[A.Compose] = None,
        is_train:  bool = True,
    ) -> None:
        self.df        = df.reset_index(drop=True)
        self.data_dir  = data_dir
        self.transform = transform
        self.is_train  = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _resolve(self, rel_path: str) -> Path:
        """Robust path resolution — handles doubled-folder mirror quirk."""
        c = self.data_dir / rel_path
        if c.exists():
            return c
        parts = Path(rel_path).parts
        if len(parts) > 1:
            d = self.data_dir / parts[0] / rel_path
            if d.exists():
                return d
        f = self.data_dir / Path(rel_path).name
        if f.exists():
            return f
        return c

    def __getitem__(self, idx: int) -> Dict:
        row = self.df.iloc[idx]

        # Load image
        img_path = self._resolve(row['image_path'])
        try:
            image = np.array(Image.open(img_path).convert('RGB'))
        except Exception as e:
            print(f'Warning: cannot load {img_path}: {e}')
            image = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE, 3), dtype=np.uint8)

        # Apply augmentation
        if self.transform is not None:
            image = self.transform(image=image)['image']

        out = {
            'image':      image,
            'is_digital': torch.tensor(float(row.get('is_digital', 0)),
                                       dtype=torch.float32),
            'type_idx':   torch.tensor(int(row.get('type_idx', 0)),
                                       dtype=torch.long),
            'id':         str(row['id']),
        }
        if self.is_train:
            out['label'] = torch.tensor(int(row['label']), dtype=torch.long)
        return out


def build_weighted_sampler(df: pd.DataFrame) -> WeightedRandomSampler:
    """
    Build a WeightedRandomSampler that:
    1. Balances label classes (genuine vs fraud)
    2. Up-weights under-represented (type x is_digital) strata

    Critical for cross-domain generalization — without it, common document
    types dominate training and rare types are barely seen.

    Fully vectorized via pandas — no Python loop over rows.
    SAMPLER_CAP limits samples per epoch for faster validation runs;
    set CFG.SAMPLER_CAP = -1 to use the full dataset length.
    """
    # Vectorized label weight: inverse class frequency
    label_w = df['label'].map(
        (1.0 / df['label'].value_counts()).to_dict()
    ).values.astype(float)

    # Vectorized stratum weight: inverse (type x is_digital) group frequency
    strat_w = (
        1.0 / df.groupby(['type', 'is_digital'])['label']
        .transform('count')
        .values.astype(float)
    )

    weights = torch.tensor(label_w * strat_w, dtype=torch.double)

    n_samples = (
        len(weights) if CFG.SAMPLER_CAP <= 0
        else min(len(weights), CFG.SAMPLER_CAP)
    )

    return WeightedRandomSampler(
        weights,
        num_samples=n_samples,
        replacement=True,
    )


print('FREUIDDataset and build_weighted_sampler defined.')

## 8. Model Architecture

In [ ]:
class FREUIDModel(nn.Module):
    """
    EfficientNet-B4 backbone + lightweight metadata conditioning head.

    Architecture:
      backbone (tf_efficientnet_b4)    → GAP    → [N, feat_dim]
      doc_embedding(type_idx)          →         → [N, doc_emb_dim]
      is_digital (scalar)              →         → [N, 1]
      concat → LayerNorm → Dropout → Linear(256) → GELU → Linear(1) → logit

    Weights: loaded from Kaggle model hub (offline, no network call).
    License: Apache-2.0 via timm — OSI-compatible, prize-eligible.

    NOTE on type embedding: the private test set contains 2 unseen document
    types. Those receive the <UNK> embedding (index 0, padding_idx=0).
    The model must rely primarily on visual features, not type embedding.
    """

    def __init__(
        self,
        backbone_name: str   = CFG.BACKBONE,
        pretrained:    bool  = CFG.PRETRAINED,
        weights_path:  str   = CFG.WEIGHTS_PATH,
        n_doc_types:   int   = CFG.N_DOC_TYPES,
        doc_emb_dim:   int   = CFG.DOC_EMB_DIM,
        drop_rate:     float = CFG.DROP_RATE,
        use_metadata:  bool  = CFG.USE_METADATA,
    ) -> None:
        super().__init__()
        self.use_metadata = use_metadata

        # ── Backbone ─────────────────────────────────────────────────────────
        # Build architecture without downloading weights (pretrained=False),
        # then load the local .pth file from Kaggle model hub manually.
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=False,   # never fetch from network
            num_classes=0,      # remove classifier head (GAP output only)
            global_pool='avg',
        )

        if pretrained:
            wp = Path(weights_path)
            if wp.exists():
                state_dict = torch.load(wp, map_location='cpu')
                missing, unexpected = self.backbone.load_state_dict(
                    state_dict, strict=False
                )
                print(f'  Backbone weights loaded: {wp.name}')
                # Missing keys: classifier head keys (expected — we removed it)
                # Unexpected keys: none expected; flag if present
                if missing:
                    print(f'  Missing keys   : {len(missing)} '
                          f'(expected — head removed with num_classes=0)')
                if unexpected:
                    print(f'  Unexpected keys: {len(unexpected)} — inspect if > 5')
            else:
                print(f'  WARNING: weights not found at {weights_path}')
                print('  Training from random init — validation metrics will be weak.')

        feat_dim = self.backbone.num_features

        # ── Metadata embedding ────────────────────────────────────────────────
        if use_metadata:
            self.doc_embedding = nn.Embedding(
                num_embeddings=n_doc_types,
                embedding_dim=doc_emb_dim,
                padding_idx=0,   # 0 = <UNK> / unseen types → zero embedding
            )
            meta_dim = doc_emb_dim + 1   # +1 for is_digital scalar
        else:
            meta_dim = 0

        # ── Classifier head ───────────────────────────────────────────────────
        in_dim = feat_dim + meta_dim
        self.head = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Dropout(drop_rate),
            nn.Linear(in_dim, 256),
            nn.GELU(),
            nn.Dropout(drop_rate / 2),
            nn.Linear(256, 1),
        )

        # Xavier init for the new head layers
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(
        self,
        image:      torch.Tensor,   # (N, C, H, W)
        is_digital: torch.Tensor,   # (N,) float
        type_idx:   torch.Tensor,   # (N,) long
    ) -> torch.Tensor:              # (N,) logits
        feats = self.backbone(image)   # (N, feat_dim)
        if self.use_metadata:
            doc_emb = self.doc_embedding(type_idx)    # (N, doc_emb_dim)
            feats   = torch.cat(
                [feats, doc_emb, is_digital.unsqueeze(1)], dim=1
            )
        return self.head(feats).squeeze(1)   # (N,) logits


# ── Quick sanity check ────────────────────────────────────────────────────────
_m = FREUIDModel()
total     = sum(p.numel() for p in _m.parameters())
trainable = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'\nModel params — Total: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M')

_m = _m.to(DEVICE)
_img  = torch.randn(2, 3, CFG.IMG_SIZE, CFG.IMG_SIZE).to(DEVICE)
_dig  = torch.tensor([0.0, 1.0]).to(DEVICE)
_tidx = torch.tensor([1, 2]).to(DEVICE)
with torch.no_grad():
    _out = _m(_img, _dig, _tidx)
print(f'Forward pass OK — output shape: {_out.shape}, values: {_out}')
del _m, _img, _dig, _tidx, _out
torch.cuda.empty_cache()

## 9. Loss Functions

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal loss for binary classification (Lin et al., 2017).

    FL = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Down-weights easy negatives (well-classified genuine samples) so
    training focuses on hard examples (subtle GenAI edits, borderline
    recaptured frauds). Addresses class imbalance via alpha weighting.
    """

    def __init__(
        self,
        alpha: float = CFG.FOCAL_ALPHA,
        gamma: float = CFG.FOCAL_GAMMA,
    ) -> None:
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(
        self, logits: torch.Tensor, targets: torch.Tensor
    ) -> torch.Tensor:
        targets = targets.float()
        bce     = F.binary_cross_entropy_with_logits(
            logits, targets, reduction='none')
        probs   = torch.sigmoid(logits)
        p_t     = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - p_t) ** self.gamma * bce).mean()


class PartialAUCLoss(nn.Module):
    """
    Surrogate partial AUC loss targeting the low-FPR region [0, fpr_max].

    Motivation: FREUID Score penalizes failures specifically at 1% BPCER.
    By maximizing pairwise ranking between attacks and the HARDEST genuine
    samples (the top fpr_max% by score — the ones that set the 1% threshold),
    we directly target the operating point the metric cares about.

    Based on: Yan et al., 'Optimizing AUROC and AUPRC' (NeurIPS 2022),
    simplified to a margin-based pairwise loss.
    """

    def __init__(
        self,
        fpr_max: float = CFG.FPR_MAX,
        margin:  float = 1.0,
    ) -> None:
        super().__init__()
        self.fpr_max = fpr_max
        self.margin  = margin

    def forward(
        self, logits: torch.Tensor, targets: torch.Tensor
    ) -> torch.Tensor:
        probs   = torch.sigmoid(logits)
        targets = targets.float()
        attack  = probs[targets == 1]
        genuine = probs[targets == 0]

        if len(attack) == 0 or len(genuine) == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        # Retain only the top fpr_max% genuine scores (hardest genuine samples)
        # These are the genuine samples most likely to set the 1% BPCER threshold
        k = max(1, int(math.ceil(self.fpr_max * len(genuine))))
        hard_genuine, _ = torch.topk(genuine, k=k)

        # Pairwise margin loss: for every (attack, hard_genuine) pair,
        # penalize if attack score ≤ genuine score + margin
        diff = attack.unsqueeze(1) - hard_genuine.unsqueeze(0)   # (n_att, k)
        return F.relu(self.margin - diff).mean()


class CombinedLoss(nn.Module):
    """
    Blended loss: (1 - w) * FocalLoss + w * PartialAUCLoss

    FocalLoss  : handles class imbalance, focuses on hard examples
    PartialAUC : directly optimizes the 1% BPCER operating region
    """

    def __init__(
        self,
        pauc_weight: float = CFG.PAUC_WEIGHT,
        fpr_max:     float = CFG.FPR_MAX,
    ) -> None:
        super().__init__()
        self.focal  = FocalLoss()
        self.pauc   = PartialAUCLoss(fpr_max=fpr_max)
        self.w      = pauc_weight

    def forward(
        self, logits: torch.Tensor, targets: torch.Tensor
    ) -> Tuple[torch.Tensor, Dict]:
        focal_loss = self.focal(logits, targets)
        pauc_loss  = self.pauc(logits, targets)
        total      = (1 - self.w) * focal_loss + self.w * pauc_loss
        return total, {
            'focal_loss': focal_loss.item(),
            'pauc_loss':  pauc_loss.item(),
        }


def get_loss_fn() -> nn.Module:
    """Return the appropriate loss function based on CFG.LOSS_TYPE."""
    if CFG.LOSS_TYPE == 'focal':      return FocalLoss()
    if CFG.LOSS_TYPE == 'pauc_focal': return CombinedLoss()
    if CFG.LOSS_TYPE == 'bce':        return nn.BCEWithLogitsLoss()
    raise ValueError(f'Unknown loss type: {CFG.LOSS_TYPE}')


print('Loss functions defined: FocalLoss, PartialAUCLoss, CombinedLoss')

## 10. Training Pipeline

In [ ]:
def build_scheduler(
    optimizer,
    n_epochs:        int,
    steps_per_epoch: int,
):
    """
    Linear warmup + cosine annealing LR schedule, operating on steps.

    - Warmup phase (first WARMUP_EPOCHS): LR linearly increases from
      LR_MIN to LR, preventing large gradient updates early in training.
    - Cosine phase (remaining epochs): LR smoothly decays from LR to LR_MIN,
      allowing fine-grained convergence without oscillation.
    """
    total_steps  = n_epochs * steps_per_epoch
    warmup_steps = CFG.WARMUP_EPOCHS * steps_per_epoch

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            # Linear warmup
            return (CFG.LR_MIN / CFG.LR +
                    (1.0 - CFG.LR_MIN / CFG.LR) *
                    step / max(1, warmup_steps))
        # Cosine decay
        progress = ((step - warmup_steps) /
                    max(1, total_steps - warmup_steps))
        return (CFG.LR_MIN / CFG.LR +
                0.5 * (1.0 - CFG.LR_MIN / CFG.LR) *
                (1.0 + math.cos(math.pi * progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler,
    loss_fn:   nn.Module,
    scaler:    Optional[torch.cuda.amp.GradScaler],
    epoch:     int,
) -> Dict[str, float]:
    """
    Single training epoch with AMP, gradient accumulation, and gradient clipping.

    FREUID metric is intentionally NOT computed here — computing it requires
    O(n^2) threshold sweeps over all training predictions, which takes
    10-20 minutes per epoch for 34k samples. Training is monitored with
    loss + AUC-ROC only (both cheap). FREUID is computed on validation only.
    """
    model.train()
    total_loss   = 0.0
    all_logits   = []
    all_labels   = []
    n_batches    = len(loader)
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        img  = batch['image'].to(DEVICE)
        dig  = batch['is_digital'].to(DEVICE)
        tidx = batch['type_idx'].to(DEVICE)
        lbl  = batch['label'].to(DEVICE)

        # Forward pass with AMP
        with torch.cuda.amp.autocast(
                enabled=CFG.AMP and scaler is not None):
            logits = model(img, dig, tidx)
            if isinstance(loss_fn, CombinedLoss):
                loss, _ = loss_fn(logits, lbl)
            else:
                loss = loss_fn(logits, lbl.float())
            loss = loss / CFG.GRAD_ACCUM

        # Backward pass
        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        # Optimizer step every GRAD_ACCUM batches
        if (step + 1) % CFG.GRAD_ACCUM == 0 or (step + 1) == n_batches:
            if scaler is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item() * CFG.GRAD_ACCUM
        all_logits.append(logits.detach().cpu().float())
        all_labels.append(lbl.detach().cpu())

    logits_np = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy()
    scores_np = 1.0 / (1.0 + np.exp(-logits_np))

    # Cheap training metrics only
    try:
        trn_auc = roc_auc_score(labels_np, scores_np)
    except Exception:
        trn_auc = float('nan')

    return {
        'loss':       total_loss / n_batches,
        'auc_roc':    trn_auc,
        'freuid':     float('nan'),    # computed on val only
        'audet':      float('nan'),
        'apcer_1pct': float('nan'),
    }


@torch.no_grad()
def validate(
    model:   nn.Module,
    loader:  DataLoader,
    loss_fn: nn.Module,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray, np.ndarray]:
    """
    Validation loop — computes full FREUID Score on the validation fold.

    Returns:
      metrics   : dict with freuid, audet, apcer_1pct, auc_roc, loss
      scores_np : sigmoid probabilities (N,)
      logits_np : raw logits (N,) — needed for temperature scaling
      labels_np : ground truth labels (N,)
    """
    model.eval()
    total_loss = 0.0
    all_logits = []
    all_labels = []

    for batch in loader:
        img  = batch['image'].to(DEVICE)
        dig  = batch['is_digital'].to(DEVICE)
        tidx = batch['type_idx'].to(DEVICE)
        lbl  = batch['label'].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=CFG.AMP):
            logits = model(img, dig, tidx)
            if isinstance(loss_fn, CombinedLoss):
                loss, _ = loss_fn(logits, lbl)
            else:
                loss = loss_fn(logits, lbl.float())

        total_loss += loss.item()
        all_logits.append(logits.cpu().float())
        all_labels.append(lbl.cpu())

    logits_np = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy()
    scores_np = 1.0 / (1.0 + np.exp(-logits_np))

    metrics        = compute_freuid_score(labels_np, scores_np)
    metrics['loss'] = total_loss / len(loader)
    return metrics, scores_np, logits_np, labels_np


print('build_scheduler, train_one_epoch, validate defined.')

In [ ]:
def train_fold(fold: int = 0) -> Tuple:
    """
    Train on one fold of the stratified cross-validation split.

    Features:
    - Resume-from-checkpoint: if a checkpoint for this fold already exists,
      training continues from where it left off rather than restarting.
    - Fast val subsampling: validation runs on 5,000 samples per epoch for
      speed monitoring; full val is run once at the end on all val samples.
    - Checkpoint safety copy: every new best is copied to output root so
      it persists on Kaggle commit even if the session crashes mid-training.

    Returns:
      model      : trained FREUIDModel with best checkpoint loaded
      calibrator : fitted temperature scaler (or None if CALIBRATE=False)
      history    : list of per-epoch metric dicts
      best_state : dict with val scores, logits, labels at best FREUID epoch
    """
    print(f'\n{"="*60}')
    print(f' Training Fold {fold} / {n_folds_safe - 1}')
    print(f'{"="*60}')

    trn_df = train_df[train_df['fold'] != fold].reset_index(drop=True)
    val_df = train_df[train_df['fold'] == fold].reset_index(drop=True)
    print(f'Train: {len(trn_df)} | Val: {len(val_df)}')

    # ── Datasets ──────────────────────────────────────────────────────────────
    # build_val_transform for training: no augmentation during validation run.
    # Switch to build_train_transform for the final competition run.
    trn_ds = FREUIDDataset(trn_df, data_dir, build_val_transform(), is_train=True)

    # Fast val: subsample 5k for per-epoch monitoring (cuts val time ~7x)
    val_df_fast = val_df.sample(
        n=min(5000, len(val_df)), random_state=SEED
    ).reset_index(drop=True)
    val_ds_fast = FREUIDDataset(
        val_df_fast, data_dir, build_val_transform(), is_train=True)

    # Full val dataset: used only at end for final metrics
    val_ds_full = FREUIDDataset(
        val_df, data_dir, build_val_transform(), is_train=True)

    sampler    = build_weighted_sampler(trn_df)
    trn_loader = DataLoader(
        trn_ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
        num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY, drop_last=True)
    val_loader_fast = DataLoader(
        val_ds_fast, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)
    val_loader_full = DataLoader(
        val_ds_full, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)

    # ── Model & optimizer ──────────────────────────────────────────────────────
    model   = FREUIDModel().to(DEVICE)
    loss_fn = get_loss_fn()

    param_groups = [
        {'params': model.backbone.parameters(),      'lr': CFG.LR * 0.1},
        {'params': model.head.parameters(),          'lr': CFG.LR},
    ]
    if CFG.USE_METADATA:
        param_groups.append(
            {'params': model.doc_embedding.parameters(), 'lr': CFG.LR})

    optimizer = torch.optim.AdamW(param_groups, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = build_scheduler(optimizer, CFG.EPOCHS, len(trn_loader))
    scaler    = (
        torch.cuda.amp.GradScaler()
        if CFG.AMP and torch.cuda.is_available() else None)

    # ── Resume from checkpoint if available ───────────────────────────────────
    ckpt_path   = Path(CFG.CHECKPOINT_DIR) / f'fold{fold}_best.pth'
    resume_epoch = 0
    best_freuid  = float('inf')
    best_state   = None
    history      = []

    if ckpt_path.exists():
        print(f'Resuming from checkpoint: {ckpt_path}')
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state'])
        resume_epoch = ckpt['epoch'] + 1
        best_freuid  = ckpt['val_metrics']['freuid']
        best_state   = ckpt
        print(f'  Resumed from epoch {resume_epoch - 1}, '
              f'best FREUID so far: {best_freuid:.4f}')
    else:
        print('No existing checkpoint — training from scratch.')

    # ── Training loop ──────────────────────────────────────────────────────────
    for epoch in range(resume_epoch, CFG.EPOCHS):
        t0    = time.time()
        trn_m = train_one_epoch(
            model, trn_loader, optimizer, scheduler, loss_fn, scaler, epoch)

        # Fast val: 5k samples for per-epoch monitoring
        val_m, val_scores, val_logits, val_labels = validate(
            model, val_loader_fast, loss_fn)
        elapsed = time.time() - t0

        row = {
            'epoch':          epoch,
            'trn_loss':       trn_m['loss'],
            'trn_auc':        trn_m['auc_roc'],
            'val_loss':       val_m['loss'],
            'val_freuid':     val_m['freuid'],
            'val_audet':      val_m['audet'],
            'val_apcer_1pct': val_m['apcer_1pct'],
            'val_auc':        val_m['auc_roc'],
            'lr':             optimizer.param_groups[0]['lr'],
            'elapsed_s':      elapsed,
        }
        history.append(row)

        print(
            f'  Ep {epoch:02d} | '
            f'trn_loss={trn_m["loss"]:.4f} trn_auc={trn_m["auc_roc"]:.4f} | '
            f'val_freuid={val_m["freuid"]:.4f} '
            f'(audet={val_m["audet"]:.4f}, '
            f'apcer@1%={val_m["apcer_1pct"]:.4f}) | '
            f'val_auc={val_m["auc_roc"]:.4f} | '
            f't={elapsed:.0f}s'
        )

        # Save checkpoint on every new best
        if val_m['freuid'] < best_freuid:
            best_freuid = val_m['freuid']
            best_state  = {
                'model_state': model.state_dict(),
                'val_scores':  val_scores,
                'val_logits':  val_logits,
                'val_labels':  val_labels,
                'val_metrics': val_m,
                'epoch':       epoch,
                'fold':        fold,
            }
            # Primary checkpoint
            torch.save(best_state, ckpt_path)
            # Safety copy to output root — persists on Kaggle commit
            import shutil as _shutil
            _shutil.copy2(ckpt_path, f'{CFG.OUTPUT_DIR}/fold{fold}_best.pth')
            print(f'    ✓ New best FREUID={best_freuid:.4f} — checkpoint saved.')

    # ── Final full validation ──────────────────────────────────────────────────
    # Load best checkpoint and run on the complete val set for final metrics
    print('\nRunning final full validation...')
    model.load_state_dict(best_state['model_state'])
    val_m_full, val_scores_full, val_logits_full, val_labels_full = validate(
        model, val_loader_full, loss_fn)
    print(f'Full val FREUID={val_m_full["freuid"]:.4f} '
          f'(audet={val_m_full["audet"]:.4f}, '
          f'apcer@1%={val_m_full["apcer_1pct"]:.4f}) | '
          f'val_auc={val_m_full["auc_roc"]:.4f}')

    # Update best_state with full val results
    best_state.update({
        'val_scores':  val_scores_full,
        'val_logits':  val_logits_full,
        'val_labels':  val_labels_full,
        'val_metrics': val_m_full,
    })
    torch.save(best_state, ckpt_path)
    import shutil as _shutil
    _shutil.copy2(ckpt_path, f'{CFG.OUTPUT_DIR}/fold{fold}_best.pth')
    print(f'Final checkpoint saved with full val metrics.')

    # ── Calibration ───────────────────────────────────────────────────────────
    calibrator = None
    if CFG.CALIBRATE:
        print('Fitting calibrator on full val fold...')
        calibrator = get_calibrator()
        calibrator.fit(best_state['val_logits'], best_state['val_labels'])
        cal_scores = calibrator.transform(best_state['val_logits'])
        m_before   = compute_freuid_score(
            best_state['val_labels'], best_state['val_scores'])
        m_after    = compute_freuid_score(
            best_state['val_labels'], cal_scores)
        print(f'  FREUID before calibration: {m_before["freuid"]:.4f}')
        print(f'  FREUID after  calibration: {m_after["freuid"]:.4f}')

    # ── DET curve ─────────────────────────────────────────────────────────────
    plot_det_curve(
        best_state['val_labels'],
        best_state['val_scores'],
        title=f'Fold {fold} DET Curve',
    )

    return model, calibrator, history, best_state


print('train_fold() defined.')


In [ ]:
# ── Full end-to-end execution ────────────────────────────────────────────────
# This cell runs the complete pipeline in sequence:
#   1. Train fold 0
#   2. Safety-copy checkpoint
#   3. Plot training curves
#   4. Val breakdown by doc type
#   5. Generate fold 0 submission
#   6. Train all folds (ensemble)
#   7. Generate ensemble submission
#   8. Log ablation result
#
# When run via Save Version → Save & Run All, Kaggle executes this
# sequentially and commits all outputs (checkpoints + CSVs) on completion.
# You can close the browser — the committed run continues on Kaggle servers.

# ── Step 1: Train fold 0 ─────────────────────────────────────────────────────
model, calibrator, history, best_state = train_fold(fold=0)

# ── Step 2: Plot training curves ─────────────────────────────────────────────
plot_training_history(history)

# ── Step 3: Val breakdown by document type and modality ──────────────────────
val_df_fold0 = train_df[train_df['fold'] == 0].reset_index(drop=True)
breakdown = evaluate_breakdown(val_df_fold0, best_state['val_scores'])
print('\n=== Val Breakdown by Group ===')
print(breakdown[
    ['group', 'subset', 'n', 'freuid', 'audet', 'apcer_1pct', 'auc_roc']
].sort_values('freuid').to_string(index=False))
breakdown.to_csv(f'{CFG.OUTPUT_DIR}/val_breakdown_fold0.csv', index=False)

# ── Step 4: Generate fold 0 submission ───────────────────────────────────────
# Submit this immediately after the run completes to lock in tie-breaker.
print('\nGenerating fold 0 submission...')
test_scores_fold0 = predict(model, test_df, calibrator, use_tta=True)
generate_submission(test_scores_fold0, test_df, 'submission_fold0.csv')

# ── Step 5: Train all folds and build ensemble ────────────────────────────────
print('\nStarting full ensemble training...')
fold_results, oof_df = train_all_folds()

# ── Step 6: Generate ensemble submission ─────────────────────────────────────
print('\nGenerating ensemble submission...')
ensemble_scores = ensemble_predict(fold_results, test_df, method='rank_avg')
generate_submission(ensemble_scores, test_df, 'submission_ensemble.csv')

# ── Step 7: Log ablation result ───────────────────────────────────────────────
log_ablation(
    run_name='effb4-pauc-fold0',
    labels=best_state['val_labels'],
    scores=best_state['val_scores'],
    config_notes=(
        f'EfficientNet-B4 tf variant, pAUC+Focal loss, '
        f'no augmentation, IMG_SIZE={CFG.IMG_SIZE}, '
        f'SAMPLER_CAP={CFG.SAMPLER_CAP}'
    )
)
show_ablation()

print('\n=== All done! ==='
      '\nOutputs saved to /kaggle/working/:'
      '\n  fold0_best.pth         — best fold 0 checkpoint'
      '\n  submission_fold0.csv   — fold 0 submission (submit first)'
      '\n  submission_ensemble.csv — ensemble submission (submit second)'
      '\n  val_breakdown_fold0.csv — breakdown by doc type'
      '\n  training_curves.png    — loss/metric curves'
      '\n  oof_scores.csv         — out-of-fold scores for report')


## 11. Score Calibration

In [ ]:
class TemperatureScaler:
    """
    Post-hoc temperature scaling (Guo et al., 2017).

    Learns a single scalar T > 0 such that sigmoid(logit / T) is better
    calibrated than sigmoid(logit). Fit on the validation fold only
    (never on training data — that would cause data leakage).

    Why mandatory for this competition:
    APCER@1%BPCER depends on the score distribution near a specific
    decision threshold. A model with perfect ranking (AUC=1.0) but
    poorly calibrated probabilities can still fail at the 1% BPCER point
    because the scores are clustered in the wrong region of [0, 1].
    """

    def __init__(self) -> None:
        self.temperature = 1.0

    def fit(
        self, logits: np.ndarray, labels: np.ndarray
    ) -> 'TemperatureScaler':
        """Find T that minimizes NLL on (logits, labels)."""
        def nll(t: np.ndarray) -> float:
            t = float(t[0])
            if t <= 0:
                return 1e9
            p = np.clip(1.0 / (1.0 + np.exp(-logits / t)), 1e-7, 1 - 1e-7)
            return -np.mean(
                labels * np.log(p) + (1 - labels) * np.log(1 - p)
            )
        res = scipy.optimize.minimize(
            nll, [1.0], method='L-BFGS-B', bounds=[(0.05, 20.0)])
        self.temperature = float(res.x[0])
        print(f'  Temperature = {self.temperature:.4f}')
        return self

    def transform(self, logits: np.ndarray) -> np.ndarray:
        """Apply temperature scaling and return probabilities in [0, 1]."""
        return (1.0 / (1.0 + np.exp(-logits / self.temperature))).astype(
            np.float32)


class IsotonicCalibrator:
    """
    Isotonic regression calibration — non-parametric, monotone.
    More flexible than temperature scaling but needs more val data.
    Fit on val fold only.
    """

    def __init__(self) -> None:
        self.iso = IsotonicRegression(out_of_bounds='clip')

    def fit(
        self, logits: np.ndarray, labels: np.ndarray
    ) -> 'IsotonicCalibrator':
        scores = 1.0 / (1.0 + np.exp(-logits))
        self.iso.fit(scores, labels)
        return self

    def transform(self, logits: np.ndarray) -> np.ndarray:
        scores = 1.0 / (1.0 + np.exp(-logits))
        return self.iso.predict(scores).astype(np.float32)


def get_calibrator():
    """Return calibrator based on CFG.CALIB_METHOD."""
    if CFG.CALIB_METHOD == 'temperature': return TemperatureScaler()
    if CFG.CALIB_METHOD == 'isotonic':    return IsotonicCalibrator()
    raise ValueError(f'Unknown calibration method: {CFG.CALIB_METHOD}')


def plot_calibration_curve(
    scores: np.ndarray,
    labels: np.ndarray,
    title:  str = 'Calibration Curve',
    n_bins: int = 10,
) -> None:
    """Reliability diagram: predicted probability vs actual fraction positive."""
    prob_true, prob_pred = calibration_curve(labels, scores, n_bins=n_bins)
    plt.figure(figsize=(5, 5))
    plt.plot(prob_pred, prob_true, 'b-o', label='Model')
    plt.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    plt.xlabel('Mean predicted probability')
    plt.ylabel('Fraction of positives')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


print('TemperatureScaler, IsotonicCalibrator, get_calibrator defined.')

## 12. Inference & Submission

In [ ]:
# TTA transforms: original + horizontal flip
# Averaging logits across TTA before calibrating stabilises scores
# near the decision boundary without distorting calibration.
TTA_TRANSFORMS = [
    build_val_transform(),
    A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.HorizontalFlip(p=1.0),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ]),
]


@torch.no_grad()
def predict(
    model:      nn.Module,
    df:         pd.DataFrame,
    calibrator  = None,
    use_tta:    bool = True,
) -> np.ndarray:
    """
    Run inference on df and return calibrated scores in [0, 1].

    Strategy:
    1. Run model on each TTA transform, collect raw logits
    2. Average logits across TTA (not probabilities — avoids Jensen's inequality)
    3. Apply calibrator to mean logits → final scores in [0, 1]

    Args:
      model      : trained FREUIDModel (best checkpoint)
      df         : DataFrame with image_path, is_digital, type_idx columns
      calibrator : fitted TemperatureScaler or IsotonicCalibrator (or None)
      use_tta    : whether to average over TTA transforms

    Returns:
      scores : (N,) array of fraud probabilities in [0, 1]
    """
    model.eval()
    transforms  = TTA_TRANSFORMS if use_tta else [build_val_transform()]
    all_logits  = []

    for tfm in transforms:
        ds = FREUIDDataset(df, data_dir, tfm, is_train=False)
        loader = DataLoader(
            ds,
            batch_size=CFG.BATCH_SIZE * 2,
            shuffle=False,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=CFG.PIN_MEMORY,
        )
        tfm_logits = []
        for batch in loader:
            with torch.cuda.amp.autocast(enabled=CFG.AMP):
                out = model(
                    batch['image'].to(DEVICE),
                    batch['is_digital'].to(DEVICE),
                    batch['type_idx'].to(DEVICE),
                )
            tfm_logits.append(out.cpu().float().numpy())
        all_logits.append(np.concatenate(tfm_logits))

    # Average logits across TTA transforms
    mean_logits = np.mean(all_logits, axis=0)

    # Apply calibration
    if calibrator is not None:
        return calibrator.transform(mean_logits)
    return (1.0 / (1.0 + np.exp(-mean_logits))).astype(np.float32)


def generate_submission(
    scores:   np.ndarray,
    df:       pd.DataFrame,
    filename: str = 'submission.csv',
) -> pd.DataFrame:
    """
    Generate submission file in the required format: id, score.

    IMPORTANT: the competition dataset card example shows column 'label',
    but the actual required column is 'score'. We cross-check against
    sample_submission.csv at runtime and warn if there is a discrepancy.
    Scores are validated to be in [0, 1] before writing.
    """
    assert len(scores) == len(df), \
        f'Length mismatch: {len(scores)} scores vs {len(df)} rows'
    assert np.all((scores >= 0) & (scores <= 1)), \
        'All scores must be in [0, 1]'

    # Determine correct score column name from sample_submission.csv
    score_col = 'score'
    if not sub_df.empty:
        expected = set(sub_df.columns) - {'id'}
        if expected and score_col not in expected:
            actual = next(iter(expected))
            print(f'WARNING: sample_submission.csv uses column "{actual}", '
                  f'not "score". Using "{actual}" to match actual format.')
            score_col = actual

    sub = pd.DataFrame({'id': df['id'].values, score_col: scores})
    out_path = Path(CFG.OUTPUT_DIR) / filename
    sub.to_csv(out_path, index=False)

    print(f'Submission saved : {out_path}')
    print(f'Rows             : {len(sub)}')
    print(f'Score stats      : min={scores.min():.4f} '
          f'max={scores.max():.4f} '
          f'mean={scores.mean():.4f} '
          f'median={np.median(scores):.4f}')
    print(sub.head())
    return sub


print('predict() and generate_submission() defined.')

## 13. Ensemble Utilities

In [ ]:
def rank_average(score_arrays: List[np.ndarray]) -> np.ndarray:
    """
    Rank averaging: convert each model's scores to percentile ranks,
    then average across models.

    More robust than averaging raw probabilities when models have different
    calibration scales or score distributions. Particularly useful when
    combining models trained with different backbones or loss functions.
    """
    ranks = [scipy.stats.rankdata(s) / len(s) for s in score_arrays]
    return np.mean(ranks, axis=0).astype(np.float32)


def train_all_folds() -> Tuple[List, pd.DataFrame]:
    """
    Train all n_folds_safe folds and collect out-of-fold (OOF) scores.

    OOF scores cover the full training set without leakage and give
    the most reliable estimate of true generalization performance.

    Returns:
      fold_results : list of (model, calibrator, history) per fold
      oof_df       : DataFrame with OOF scores for every training sample
    """
    fold_results = []
    oof_records  = []

    for fold in range(n_folds_safe):
        model, calibrator, history, best_state = train_fold(fold=fold)

        # Collect OOF predictions for this fold
        val_df_fold = train_df[
            train_df['fold'] == fold].reset_index(drop=True)
        oof_scores = predict(model, val_df_fold, calibrator, use_tta=False)

        for i, (_, row) in enumerate(val_df_fold.iterrows()):
            oof_records.append({
                'id':         row['id'],
                'label':      row['label'],
                'is_digital': row['is_digital'],
                'type':       row['type'],
                'oof_score':  float(oof_scores[i]),
                'fold':       fold,
            })
        fold_results.append((model, calibrator, history))

    oof_df = pd.DataFrame(oof_records)

    # OOF FREUID Score — most reliable metric for the academic report
    m = compute_freuid_score(
        oof_df['label'].values, oof_df['oof_score'].values)
    print('\n=== OOF FREUID Score ===')
    for k, v in m.items():
        print(f'  {k:15s}: {v:.4f}')

    # OOF breakdown by is_digital and document type
    oof_df_eval = oof_df.rename(columns={'oof_score': 'score'})
    breakdown   = evaluate_breakdown(
        oof_df.assign(label=oof_df['label']),
        oof_df['oof_score'].values,
    )
    print('\n=== OOF Breakdown ===')
    print(breakdown[
        ['group', 'subset', 'n', 'freuid', 'audet', 'apcer_1pct', 'auc_roc']
    ].sort_values('freuid').to_string(index=False))

    # Save OOF scores for academic report
    oof_df.to_csv(f'{CFG.OUTPUT_DIR}/oof_scores.csv', index=False)

    return fold_results, oof_df


def ensemble_predict(
    fold_results: List,
    df:           pd.DataFrame,
    method:       str = 'rank_avg',   # 'rank_avg' | 'avg'
) -> np.ndarray:
    """
    Ensemble predictions from multiple fold models.

    method='rank_avg' is recommended — it's more robust to calibration
    differences between folds than averaging raw probabilities.
    """
    all_scores = [
        predict(model, df, calibrator, use_tta=True)
        for model, calibrator, _ in fold_results
    ]
    if method == 'rank_avg':
        return rank_average(all_scores)
    return np.mean(all_scores, axis=0).astype(np.float32)


print('rank_average, train_all_folds, ensemble_predict defined.')

## 14. Visualization & Ablation Logger

In [ ]:
def plot_training_history(history: List[Dict]) -> None:
    """Plot training and validation metrics over epochs."""
    hist = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Loss
    axes[0].plot(hist['epoch'], hist['trn_loss'], label='Train', marker='o', ms=3)
    axes[0].plot(hist['epoch'], hist['val_loss'],  label='Val',   marker='o', ms=3)
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    # FREUID components
    axes[1].plot(hist['epoch'], hist['val_freuid'],
                 label='FREUID ↓', color='red',    marker='o', ms=3)
    axes[1].plot(hist['epoch'], hist['val_audet'],
                 label='AuDET ↓',  color='blue',   marker='s', ms=3, ls='--')
    axes[1].plot(hist['epoch'], hist['val_apcer_1pct'],
                 label='APCER@1%', color='orange', marker='^', ms=3, ls=':')
    axes[1].set_title('Val FREUID Components')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    # AUC
    axes[2].plot(hist['epoch'], hist['trn_auc'], label='Train AUC', marker='o', ms=3)
    axes[2].plot(hist['epoch'], hist['val_auc'],  label='Val AUC',   marker='o', ms=3)
    axes[2].set_title('AUC-ROC')
    axes[2].set_xlabel('Epoch')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(f'{CFG.OUTPUT_DIR}/training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()


# ── Ablation logger ───────────────────────────────────────────────────────────
# Use this to compare experiment variants for the academic report.
# Call log_ablation() after each experiment, show_ablation() at the end.
ABLATION_LOG: List[Dict] = []

def log_ablation(
    run_name:     str,
    labels:       np.ndarray,
    scores:       np.ndarray,
    config_notes: str = '',
) -> None:
    """Log results of one ablation experiment."""
    m = compute_freuid_score(labels, scores)
    ABLATION_LOG.append({
        'run':       run_name,
        'notes':     config_notes,
        'FREUID↓':   round(m['freuid'],     4),
        'AuDET↓':    round(m['audet'],      4),
        'APCER@1%↓': round(m['apcer_1pct'], 4),
        'AUC-ROC↑':  round(m['auc_roc'],    4),
        'g_audet':   round(m['g_audet'],    4),
        'g_apcer':   round(m['g_apcer'],    4),
    })


def show_ablation() -> pd.DataFrame:
    """Display ablation results sorted by FREUID Score."""
    if not ABLATION_LOG:
        print('No ablation results logged yet. Call log_ablation() first.')
        return pd.DataFrame()
    df = pd.DataFrame(ABLATION_LOG).sort_values('FREUID↓')
    print('\n=== Ablation Results (sorted by FREUID Score) ===')
    print(df.to_string(index=False))
    df.to_csv(f'{CFG.OUTPUT_DIR}/ablation_results.csv', index=False)
    print(f'Saved to {CFG.OUTPUT_DIR}/ablation_results.csv')
    return df


print('plot_training_history, log_ablation, show_ablation defined.')

## 15. Quick-Start Execution Cells

These cells are provided as **reference and manual override** — the main execution cell (Cell 25) already runs all of these automatically as part of `Save & Run All`.

Use these individually if you need to re-run a specific step without re-training (e.g. re-generate submission from a saved checkpoint, or re-plot curves after loading a checkpoint manually).

In [ ]:
# ── Manual checkpoint loader ─────────────────────────────────────────────────
# Use this if the session crashed and you want to load a saved checkpoint
# without re-running train_fold(). The checkpoint is in /kaggle/working/.

# fold = 0
# ckpt = torch.load(f'/kaggle/working/fold{fold}_best.pth', map_location=DEVICE)
# model = FREUIDModel().to(DEVICE)
# model.load_state_dict(ckpt['model_state'])
# model.eval()
# calibrator = get_calibrator()
# calibrator.fit(ckpt['val_logits'], ckpt['val_labels'])
# best_state = ckpt
# print(f'Checkpoint loaded. Best FREUID={ckpt["val_metrics"]["freuid"]:.4f} at epoch {ckpt["epoch"]}')


In [ ]:
# ── Step 2: Plot training curves (manual override) ───────────────────────────
# Already runs automatically in Cell 25. Use this to re-plot after
# loading a checkpoint manually.

# plot_training_history(history)


In [ ]:
# ── Step 3: Val breakdown by document type (manual override) ─────────────────
# Already runs automatically in Cell 25.

# val_df_fold0 = train_df[train_df['fold'] == 0].reset_index(drop=True)
# breakdown = evaluate_breakdown(val_df_fold0, best_state['val_scores'])
# print(breakdown[
#     ['group', 'subset', 'n', 'freuid', 'audet', 'apcer_1pct', 'auc_roc']
# ].sort_values('freuid').to_string(index=False))


In [ ]:
# ── Step 4: Generate fold 0 submission (manual override) ─────────────────────
# Already runs automatically in Cell 25.
# Use this to re-generate without re-training.

# test_scores = predict(model, test_df, calibrator, use_tta=True)
# generate_submission(test_scores, test_df, 'submission_fold0.csv')


In [ ]:
# ── Step 5 & 6: Ensemble training + submission (manual override) ──────────────
# Already runs automatically in Cell 25.

# fold_results, oof_df = train_all_folds()
# ensemble_scores = ensemble_predict(fold_results, test_df, method='rank_avg')
# generate_submission(ensemble_scores, test_df, 'submission_ensemble.csv')


## Output Files Reference

After `Save & Run All` completes, these files will be in the Output tab:

| File | Description |
|---|---|
| `fold0_best.pth` | Best fold 0 checkpoint — loads with manual checkpoint loader above |
| `submission_fold0.csv` | Fold 0 submission — **submit first** for tie-breaker |
| `submission_ensemble.csv` | Ensemble submission — submit second |
| `val_breakdown_fold0.csv` | FREUID breakdown by document type and modality |
| `oof_scores.csv` | Out-of-fold scores for academic report |
| `training_curves.png` | Loss and metric curves per epoch |
| `ablation_results.csv` | Ablation table for academic report |

**Submission strategy (6 days remaining):**
1. Submit `submission_fold0.csv` immediately — locks tie-breaker timestamp
2. Submit `submission_ensemble.csv` once full ensemble completes
3. Daily limit: 5 submissions — validate locally before submitting
4. Earlier submissions win ties — don't wait until the deadline